# Random Forest Image-Feature Classification — Fruit and Vehicles

This notebook completes both practice problems using the supplied CSV files:

1. **Fruit:** apple, banana, lime, orange
2. **Vehicles:** car, bus, motorcycle

For each dataset, it performs exploratory data analysis, an 80/20 stratified train/test split, a Random Forest baseline, 5-fold Stratified Cross-Validation, RandomizedSearchCV hyperparameter tuning, final evaluation, feature-importance analysis, a sample prediction, and model export.

> **Important:** These CSV files contain synthetic numeric features extracted from images; they are not raw images or pixel arrays. This notebook classifies the feature rows. It does not directly accept an image file. A real-image application would first need an image-processing/feature-extraction step.

## 1. Upload the two CSV files

Run the next cell. When prompted, select both files:

- `dataset_1_fruit_image_features.csv`
- `dataset_2_vehicle_image_features.csv`

If the files are already in the current Colab working directory, the upload prompt will be skipped.

In [ ]:
import os

required_files = [
    "dataset_1_fruit_image_features.csv",
    "dataset_2_vehicle_image_features.csv",
]

missing_files = [name for name in required_files if not os.path.exists(name)]
if missing_files:
    from google.colab import files
    print("Please upload both CSV files.")
    uploaded = files.upload()

still_missing = [name for name in required_files if not os.path.exists(name)]
if still_missing:
    raise FileNotFoundError(
        "These files were not found: " + ", ".join(still_missing)
        + ". Upload the CSV files with their original filenames and rerun this cell."
    )

print("Both datasets are ready.")

## 2. Import libraries and load the datasets

The `image_id` column is an identifier, so it is not used as a model feature. The target column is `label`.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay,
    confusion_matrix,
)
import joblib
import zipfile

RANDOM_STATE = 42
TEST_SIZE = 0.20
CV_SPLITS = 5
SEARCH_ITERATIONS = 8  # Increase to 12-15 for a broader search if runtime allows.

fruit_df = pd.read_csv("dataset_1_fruit_image_features.csv")
vehicle_df = pd.read_csv("dataset_2_vehicle_image_features.csv")

print("Fruit dataset:", fruit_df.shape)
print("Vehicle dataset:", vehicle_df.shape)

display(fruit_df.head())
display(vehicle_df.head())

## 3. Exploratory Data Analysis (EDA)

This function checks the dataset dimensions, data types, missing values, duplicate records, class balance, feature distributions, and feature correlations. The overlaid histograms help reveal whether numeric features overlap across classes.

In [ ]:
def explore_dataset(df, dataset_name):
    print("=" * 75)
    print(dataset_name)
    print("=" * 75)
    print("Shape:", df.shape)
    print("\nData types:")
    display(df.dtypes.rename("dtype").to_frame())

    print("Missing values by column:")
    display(df.isna().sum().rename("missing_count").to_frame())
    print("Duplicate complete rows:", df.duplicated().sum())
    if "image_id" in df.columns:
        print("Duplicate image IDs:", df["image_id"].duplicated().sum())

    print("Class distribution:")
    class_counts = df["label"].value_counts().sort_index()
    display(class_counts.rename("count").to_frame())

    fig, ax = plt.subplots(figsize=(7, 4))
    class_counts.plot(kind="bar", ax=ax)
    ax.set_title(f"{dataset_name}: Class Distribution")
    ax.set_xlabel("Class")
    ax.set_ylabel("Number of records")
    ax.tick_params(axis="x", rotation=0)
    plt.tight_layout()
    plt.show()

    feature_cols = [c for c in df.columns if c not in ["image_id", "label"]]
    numeric_features = df[feature_cols].select_dtypes(include=np.number).columns.tolist()
    non_numeric = [c for c in feature_cols if c not in numeric_features]
    if non_numeric:
        raise ValueError(f"Non-numeric feature columns need review: {non_numeric}")

    print("Mean feature values by class (use the histograms below to inspect overlap):")
    display(df.groupby("label")[numeric_features].mean().round(3))

    # Plot each numeric feature with a separate distribution for each label.
    fig, axes = plt.subplots(3, 4, figsize=(17, 11))
    axes = axes.ravel()
    for ax, column in zip(axes, numeric_features):
        for class_name, subset in df.groupby("label"):
            ax.hist(
                subset[column].dropna(),
                bins=25,
                alpha=0.40,
                density=True,
                label=str(class_name),
            )
        ax.set_title(column)
        ax.set_ylabel("Density")
    for ax in axes[len(numeric_features):]:
        ax.set_visible(False)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=max(2, len(labels)))
    fig.suptitle(f"{dataset_name}: Feature Distributions by Class", y=1.02, fontsize=14)
    plt.tight_layout()
    plt.show()

    corr = df[numeric_features].corr()
    fig, ax = plt.subplots(figsize=(10, 8))
    image = ax.imshow(corr, aspect="auto", vmin=-1, vmax=1)
    ax.set_xticks(range(len(numeric_features)))
    ax.set_yticks(range(len(numeric_features)))
    ax.set_xticklabels(numeric_features, rotation=90)
    ax.set_yticklabels(numeric_features)
    ax.set_title(f"{dataset_name}: Feature Correlation")
    fig.colorbar(image, ax=ax, label="Correlation")
    plt.tight_layout()
    plt.show()

    return numeric_features

fruit_features = explore_dataset(fruit_df, "Fruit Classification")
vehicle_features = explore_dataset(vehicle_df, "Vehicle Classification")

## 4. Build and evaluate a Random Forest model

For each dataset, the workflow below:

1. Uses the 12 numeric image features and excludes `image_id` and `label` from `X`.
2. Splits data into 80% training and 20% testing using `stratify=y`.
3. Measures a baseline model with 5-fold Stratified Cross-Validation on the training data.
4. Evaluates the baseline on the untouched test set.
5. Tunes five Random Forest hyperparameters using `RandomizedSearchCV` and macro-F1.
6. Evaluates the tuned model on the test set and reports class-wise precision, recall, and F1.
7. Plots the confusion matrix and top five feature importances.
8. Saves the fitted preprocessing/model pipeline and demonstrates a prediction.

The median imputer is included in a pipeline so any future missing values are learned from the training folds only. The supplied files currently have no missing values.

In [ ]:
def run_random_forest_experiment(df, dataset_name, model_filename):
    # Separate predictors and target. Never use image_id as a predictor.
    feature_cols = [c for c in df.columns if c not in ["image_id", "label"]]
    X = df[feature_cols].copy()
    y = df["label"].astype(str).copy()

    # Make sure all predictor columns are numeric.
    X = X.apply(pd.to_numeric, errors="raise")

    # 80/20 split while preserving the class proportions.
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y,
    )

    cv = StratifiedKFold(
        n_splits=CV_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    # Baseline: 300 trees, with other settings left at their defaults.
    baseline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("rf", RandomForestClassifier(
            n_estimators=300,
            random_state=RANDOM_STATE,
            n_jobs=1,
        )),
    ])

    scoring = {
        "accuracy": "accuracy",
        "precision_macro": "precision_macro",
        "recall_macro": "recall_macro",
        "f1_macro": "f1_macro",
    }
    cv_results = cross_validate(
        baseline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        return_train_score=False,
    )

    print("\n" + "=" * 75)
    print(dataset_name, "— Baseline 5-fold Cross-Validation")
    print("=" * 75)
    cv_summary = pd.DataFrame({
        "mean": {metric: cv_results[f"test_{metric}"].mean() for metric in scoring},
        "std": {metric: cv_results[f"test_{metric}"].std() for metric in scoring},
    })
    display(cv_summary.round(4))

    baseline.fit(X_train, y_train)
    baseline_pred = baseline.predict(X_test)
    baseline_accuracy = accuracy_score(y_test, baseline_pred)
    baseline_macro_f1 = f1_score(y_test, baseline_pred, average="macro")

    print("Baseline test accuracy:", round(baseline_accuracy, 4))
    print("Baseline test macro-F1:", round(baseline_macro_f1, 4))

    # Hyperparameter search. This covers all four parameters explicitly requested
    # for the vehicle task, plus min_samples_split.
    parameter_choices = {
        "rf__n_estimators": [50, 100, 150],
        "rf__max_depth": [None, 10, 20],
        "rf__min_samples_split": [2, 5, 10],
        "rf__min_samples_leaf": [1, 2, 4],
        "rf__max_features": ["sqrt", 0.7, 1.0],
    }

    search = RandomizedSearchCV(
        estimator=Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("rf", RandomForestClassifier(
                random_state=RANDOM_STATE,
                n_jobs=1,
            )),
        ]),
        param_distributions=parameter_choices,
        n_iter=SEARCH_ITERATIONS,
        scoring="f1_macro",
        cv=cv,
        random_state=RANDOM_STATE,
        n_jobs=1,
        refit=True,
        verbose=0,
    )
    search.fit(X_train, y_train)
    best_model = search.best_estimator_

    print("\nBest parameters:")
    display(pd.Series(search.best_params_, name="best_value").to_frame())
    print("Best cross-validation macro-F1:", round(search.best_score_, 4))

    # Final evaluation on the held-out test set.
    tuned_pred = best_model.predict(X_test)
    tuned_accuracy = accuracy_score(y_test, tuned_pred)
    tuned_precision = precision_score(y_test, tuned_pred, average="macro", zero_division=0)
    tuned_recall = recall_score(y_test, tuned_pred, average="macro", zero_division=0)
    tuned_macro_f1 = f1_score(y_test, tuned_pred, average="macro", zero_division=0)
    train_accuracy = accuracy_score(y_train, best_model.predict(X_train))

    comparison = pd.DataFrame(
        [
            [baseline_accuracy, baseline_macro_f1],
            [tuned_accuracy, tuned_macro_f1],
        ],
        columns=["Test accuracy", "Test macro-F1"],
        index=["Baseline Random Forest", "Tuned Random Forest"],
    )
    print("\nBaseline versus tuned model:")
    display(comparison.round(4))

    print("Tuned model training accuracy:", round(train_accuracy, 4))
    print("Tuned model test accuracy:", round(tuned_accuracy, 4))
    print("Train-test accuracy gap:", round(train_accuracy - tuned_accuracy, 4))
    if train_accuracy - tuned_accuracy > 0.10:
        print("Notice: the train-test gap is relatively large; inspect possible overfitting.")
    else:
        print("The train-test accuracy gap is not large by this simple check.")

    print("\nClassification report:")
    print(classification_report(y_test, tuned_pred, zero_division=0))

    # Confusion matrix.
    fig, ax = plt.subplots(figsize=(7, 6))
    ConfusionMatrixDisplay.from_predictions(
        y_test,
        tuned_pred,
        cmap="Blues",
        xticks_rotation=45,
        ax=ax,
    )
    ax.set_title(f"{dataset_name}: Tuned Random Forest Confusion Matrix")
    plt.tight_layout()
    plt.show()

    # Feature importance from the fitted Random Forest.
    fitted_rf = best_model.named_steps["rf"]
    importance = pd.Series(
        fitted_rf.feature_importances_,
        index=feature_cols,
        name="importance",
    ).sort_values(ascending=False)
    top_five = importance.head(5)

    print("Top five feature importances:")
    display(top_five.to_frame().round(4))

    # Plain-language interpretations based on the supplied feature names.
    # Exact pixel-extraction formulas are not included in the problem statement.
    feature_meanings = {
        "red_ratio": "Proportion of the image/object feature represented by red colour.",
        "green_ratio": "Proportion represented by green colour.",
        "yellow_ratio": "Proportion represented by yellow colour.",
        "blue_ratio": "Proportion represented by blue colour.",
        "brightness_mean": "Average brightness of the image/object region.",
        "color_saturation": "Strength or intensity of the colours.",
        "roundness": "How close the object's outline is to a round shape.",
        "elongation": "How stretched or elongated the object's shape is.",
        "edge_density": "Amount of edge/boundary detail detected.",
        "texture_entropy": "Variation or complexity in the detected texture.",
        "surface_spot_ratio": "Proportion of the surface associated with spot-like marks.",
        "size_ratio": "Relative object-size measure; its exact reference scale is not specified.",
        "mean_red": "Average intensity of the red colour channel.",
        "mean_green": "Average intensity of the green colour channel.",
        "mean_blue": "Average intensity of the blue colour channel.",
        "color_variance": "How much colour values vary across the image/object region.",
        "aspect_ratio": "Relative width-to-height proportion of the object.",
        "circularity": "How circular or compact the object's outline is.",
        "corner_density": "Amount of corner-like structure detected.",
        "wheel_like_region_count": "Estimated count of regions with wheel-like visual patterns.",
        "object_area_ratio": "Proportion of the image area occupied by the detected object.",
    }
    print("\nPlain-language interpretation of the top features:")
    for feature in top_five.index:
        print(f"- {feature}: {feature_meanings.get(feature, 'Feature meaning is not defined in this notebook.')} ")

    fig, ax = plt.subplots(figsize=(8, 5))
    top_five.sort_values().plot(kind="barh", ax=ax)
    ax.set_title(f"{dataset_name}: Top 5 Feature Importances")
    ax.set_xlabel("Random Forest importance")
    ax.set_ylabel("Feature")
    plt.tight_layout()
    plt.show()

    # Test-set example: this is a held-out feature row, not a manually invented image.
    example_row = X_test.iloc[[0]]
    example_prediction = best_model.predict(example_row)[0]
    example_result = pd.DataFrame({
        "actual_label": [y_test.iloc[0]],
        "predicted_label": [example_prediction],
        "correct": [y_test.iloc[0] == example_prediction],
    })
    print("\nPrediction for one held-out example:")
    display(example_result)

    # Save everything needed to reuse the model and preserve its feature order.
    model_bundle = {
        "model": best_model,
        "feature_columns": feature_cols,
        "classes": sorted(y.unique().tolist()),
        "dataset_name": dataset_name,
    }
    joblib.dump(model_bundle, model_filename)
    print("Saved model:", model_filename)

    return {
        "dataset_name": dataset_name,
        "model": best_model,
        "model_bundle": model_bundle,
        "model_filename": model_filename,
        "feature_columns": feature_cols,
        "test_accuracy": tuned_accuracy,
        "test_precision_macro": tuned_precision,
        "test_recall_macro": tuned_recall,
        "test_macro_f1": tuned_macro_f1,
        "baseline_test_accuracy": baseline_accuracy,
        "baseline_test_macro_f1": baseline_macro_f1,
        "best_cv_macro_f1": search.best_score_,
        "best_params": search.best_params_,
        "top_features": top_five,
        "X_test": X_test,
        "y_test": y_test,
        "test_predictions": tuned_pred,
    }

print("Experiment function is ready.")

## 5. Run the fruit and vehicle experiments

Each dataset is trained and evaluated independently. The test set is held out from both cross-validation and hyperparameter tuning.

In [ ]:
fruit_results = run_random_forest_experiment(
    fruit_df,
    dataset_name="Fruit Classification",
    model_filename="fruit_random_forest_bundle.joblib",
)

vehicle_results = run_random_forest_experiment(
    vehicle_df,
    dataset_name="Vehicle Classification",
    model_filename="vehicle_random_forest_bundle.joblib",
)

## 6. Final comparison summary

In [ ]:
summary = pd.DataFrame([
    {
        "Dataset": result["dataset_name"],
        "Baseline test accuracy": result["baseline_test_accuracy"],
        "Tuned test accuracy": result["test_accuracy"],
        "Tuned precision (macro)": result["test_precision_macro"],
        "Tuned recall (macro)": result["test_recall_macro"],
        "Tuned F1 (macro)": result["test_macro_f1"],
        "Best CV F1 (macro)": result["best_cv_macro_f1"],
    }
    for result in [fruit_results, vehicle_results]
])

display(summary.set_index("Dataset").round(4))
summary.to_csv("random_forest_metrics_summary.csv", index=False)
print("Saved metrics summary: random_forest_metrics_summary.csv")

## 7. Load a saved model and predict a new feature row

The saved bundle contains the trained pipeline and the exact feature-column order expected by the model. Supply all expected numeric features. Do not pass `image_id` or `label`.

In [ ]:
def predict_from_feature_row(bundle_path, feature_values):
    bundle = joblib.load(bundle_path)
    expected_columns = bundle["feature_columns"]

    # Check that the submitted row contains every model input feature.
    missing = [column for column in expected_columns if column not in feature_values]
    extra = [column for column in feature_values if column not in expected_columns]
    if missing:
        raise ValueError(f"Missing required features: {missing}")
    if extra:
        raise ValueError(f"Unexpected features: {extra}")

    row = pd.DataFrame([[feature_values[column] for column in expected_columns]],
                       columns=expected_columns)
    row = row.apply(pd.to_numeric, errors="raise")
    prediction = bundle["model"].predict(row)[0]
    probabilities = bundle["model"].predict_proba(row)[0]
    confidence = dict(zip(bundle["model"].named_steps["rf"].classes_, probabilities))
    return prediction, confidence

# Demonstration using one known held-out fruit row.
fruit_bundle = joblib.load("fruit_random_forest_bundle.joblib")
fruit_example_values = fruit_results["X_test"].iloc[0].to_dict()
fruit_prediction, fruit_confidence = predict_from_feature_row(
    "fruit_random_forest_bundle.joblib",
    fruit_example_values,
)
print("Fruit example prediction:", fruit_prediction)
print("Class probabilities:", {k: round(float(v), 4) for k, v in fruit_confidence.items()})

# Demonstration using one known held-out vehicle row.
vehicle_example_values = vehicle_results["X_test"].iloc[0].to_dict()
vehicle_prediction, vehicle_confidence = predict_from_feature_row(
    "vehicle_random_forest_bundle.joblib",
    vehicle_example_values,
)
print("Vehicle example prediction:", vehicle_prediction)
print("Class probabilities:", {k: round(float(v), 4) for k, v in vehicle_confidence.items()})

## 8. Download the trained models and metrics

This creates one ZIP file containing both trained model bundles and the metrics summary.

In [ ]:
output_files = [
    "fruit_random_forest_bundle.joblib",
    "vehicle_random_forest_bundle.joblib",
    "random_forest_metrics_summary.csv",
]

zip_name = "random_forest_classification_outputs.zip"
with zipfile.ZipFile(zip_name, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    for filename in output_files:
        if os.path.exists(filename):
            zipf.write(filename)

print("Created:", zip_name)
try:
    from google.colab import files
    files.download(zip_name)
except Exception:
    print("Not running in Google Colab. The ZIP is saved in the current working directory.")

## 9. Interpretation and deployment notes

### Fruit sorter

- Monitor prediction accuracy and per-class precision/recall on newly labelled fruit samples.
- Track changes in lighting, camera position, fruit variety, ripeness, and surface condition because these can change the extracted colour, shape, and texture features.
- Review low-confidence predictions and route uncertain fruit to manual inspection rather than forcing a potentially incorrect packing-bin decision.
- Re-evaluate and retrain on representative, newly labelled data when performance or feature distributions drift.

### Vehicle counter / lane allocation

- Test separately under low light, glare, rain, different camera angles, partial occlusion, and crowded scenes.
- Monitor class-wise recall and the confusion matrix, not only overall accuracy. A missed bus or a motorcycle misclassified as a car can affect counting and lane allocation.
- Track changes in feature distributions as camera placement, road conditions, and vehicle types change.
- Send uncertain or out-of-distribution cases for review and periodically retrain using recent labelled camera samples.

### How to report the result honestly

- Report the held-out **test** metrics and the cross-validation mean; do not report only training accuracy.
- A 100% test score is not guaranteed and should never be forced by putting the test data into training or tuning.
- The provided data are synthetic feature tables. A high score on these tables does not establish 100% accuracy on real camera images.
- Feature importance describes how the fitted forest used the available features; it does not prove that a feature causes a class label.